# GSPO: STEM Reasoning via Verifiable Rewards

This notebook applies **GSPO** (Group Sequence Policy Optimization) with verifiable
rewards to improve STEM reasoning quality.

**Training pipeline stage:** 2 of 3 (SFT -> **GSPO** -> STaR)

**Key features:**
- `importance_sampling_level="sequence"` — GSPO sequence-level importance ratios (used in Qwen3 training)
- `loss_type="grpo"` — Standard GRPO loss (GSPO variant)
- `beta=0.0` — GSPO does not use KL regularization
- `epsilon=3e-4`, `epsilon_high=4e-4` — GSPO asymmetric clipping
- `steps_per_generation=4`, `gradient_accumulation_steps=1` — constraint: steps_per_generation = 4 × grad_accum
- G=8 completions per prompt for group-relative advantage estimation
- Two-phase training:
  - **Phase A:** Verifiable rewards only (math correctness, physics units, chemistry balance)
  - **Phase B:** Add conceptual rewards with Reasoning-as-Reward (RaR)
- Domain-specific reward functions with -0.5 penalty for wrong answers

**Domains:** Mathematics, Physics, Chemistry, Biology, Computer Science

**References:**
- [GSPO paper (arXiv 2507.18071)](https://arxiv.org/abs/2507.18071)
- [Qwen3 blog — GSPO section](https://qwenlm.github.io/blog/gspo/)
- [TRL GSPO docs](https://github.com/huggingface/trl/blob/main/docs/source/paper_index.md)

In [ ]:
# Disable gradient offloading BEFORE importing unsloth
import os
os.environ["UNSLOTH_OFFLOAD_GRADIENTS"] = "0"

# Install dependencies
!pip install -q unsloth trl peft transformers datasets
!pip install -q accelerate bitsandbytes sentencepiece protobuf
!pip install -q sympy chempy  # For verifiable reward functions

# Login to HuggingFace (needed for private SFT adapter)
from huggingface_hub import login
login()

In [ ]:
# ============================================================
# Configuration
# ============================================================

# Model — loading SFT checkpoint directly (SimPO skipped)
BASE_MODEL = "unsloth/Qwen3-4B"
SFT_CHECKPOINT = "/content/drive/MyDrive/MITS/checkpoints/sft_qwen3_4b/final_adapter"
SFT_HF_REPO = "Siesher/mits-qwen3-4b-sft"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/gspo_qwen3_4b"

# GSPO hyperparameters (from Qwen3 paper arXiv 2507.18071 + TRL docs)
# Key constraint: steps_per_generation = 4 × gradient_accumulation_steps
IMPORTANCE_SAMPLING_LEVEL = "sequence"
LOSS_TYPE = "grpo"               # GSPO uses standard "grpo" loss (NOT "dr_grpo")
BETA = 0.0                       # GSPO: no KL regularization
EPSILON = 3e-4                   # Asymmetric clipping lower bound
EPSILON_HIGH = 4e-4              # Asymmetric clipping upper bound
G = 8                            # Completions per prompt
MAX_COMPLETION = 1024
MAX_PROMPT_LENGTH = 512
LEARNING_RATE = 5e-6

# LoRA for GSPO
LORA_R = 16
LORA_ALPHA = 32
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Training
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 1  # Must satisfy: steps_per_generation = 4 × grad_accum
STEPS_PER_GENERATION = 4         # = 4 × GRADIENT_ACCUMULATION_STEPS
TOTAL_STEPS_PHASE_A = 500
TOTAL_STEPS_PHASE_B = 300
LOGGING_STEPS = 5
SAVE_STEPS = 100

# Domains
DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]

print(f"Base model: {BASE_MODEL}")
print(f"SFT checkpoint: {SFT_CHECKPOINT}")
print(f"GSPO: importance_sampling={IMPORTANCE_SAMPLING_LEVEL}, loss={LOSS_TYPE}")
print(f"GSPO: beta={BETA}, epsilon={EPSILON}/{EPSILON_HIGH}")
print(f"GSPO: steps_per_generation={STEPS_PER_GENERATION}, grad_accum={GRADIENT_ACCUMULATION_STEPS}")
print(f"G={G}, LR={LEARNING_RATE}")
print(f"Phase A: {TOTAL_STEPS_PHASE_A} steps, Phase B: {TOTAL_STEPS_PHASE_B} steps")

In [ ]:
# ============================================================
# Mount Google Drive (optional) and resolve SFT checkpoint
# ============================================================
import json
import os
import sys
from collections import Counter, defaultdict

# Try mounting Drive; fall back to local
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive mounted successfully")
except Exception as e:
    print(f"Drive mount failed ({e}), using local storage")
    OUTPUT_DIR = "/content/checkpoints/gspo_qwen3_4b"

# Resolve SFT checkpoint: Drive first, then HuggingFace
if os.path.exists(SFT_CHECKPOINT):
    print(f"SFT checkpoint found on Drive: {SFT_CHECKPOINT}")
elif SFT_HF_REPO:
    SFT_CHECKPOINT = SFT_HF_REPO
    print(f"Using SFT from HuggingFace: {SFT_CHECKPOINT}")
else:
    raise FileNotFoundError(f"SFT checkpoint not found: {SFT_CHECKPOINT}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# Load GSPO problems from HuggingFace
# ============================================================

from datasets import load_dataset

print("Loading GSPO problems from Siesher/mits-stem-training-data...")
hf_ds = load_dataset("Siesher/mits-stem-training-data", "gspo")

# Combine train+test for GSPO (we use all problems)
problems = [dict(r) for r in hf_ds["train"]] + [dict(r) for r in hf_ds["test"]]
print(f"Loaded {len(problems)} GSPO problems")

# Categorize problems
verifiable_problems = [p for p in problems if p.get("type", "verifiable") == "verifiable"]
conceptual_problems = [p for p in problems if p.get("type") == "conceptual"]

print(f"  Verifiable: {len(verifiable_problems)}")
print(f"  Conceptual: {len(conceptual_problems)}")

domain_counts = Counter(p.get("domain", "unknown") for p in problems)
for domain, count in sorted(domain_counts.items()):
    print(f"  {domain}: {count} problems")

In [ ]:
# ============================================================
# Import reward functions from training.scripts.stem_rewards
# ============================================================

try:
    from training.scripts.stem_rewards import make_reward_fn
    print("Imported make_reward_fn from training.scripts.stem_rewards")
except ImportError:
    print("WARNING: Could not import stem_rewards, defining fallback reward functions")

    import sympy
    import re

    def make_reward_fn(domain, reward_type="verifiable"):
        """Create a domain-specific reward function.

        Returns a function that takes (prompt, completion, answer) and returns a float reward.
        Correct answers get +1.0, wrong answers get -0.5 (penalty for incorrect reasoning).
        """

        def extract_boxed_answer(text):
            """Extract answer from \\boxed{...} or final numeric answer."""
            boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
            if boxed:
                return boxed[-1].strip()
            # Try extracting last number
            numbers = re.findall(r"[-+]?\d*\.?\d+", text)
            return numbers[-1] if numbers else ""

        def math_reward(prompt, completion, answer):
            """Verify math answer using SymPy equivalence."""
            extracted = extract_boxed_answer(completion)
            if not extracted or not answer:
                return 0.0
            try:
                pred = sympy.sympify(extracted)
                gold = sympy.sympify(answer)
                if sympy.simplify(pred - gold) == 0:
                    return 1.0
                return -0.5  # Wrong answer penalty
            except (sympy.SympifyError, TypeError, ValueError):
                if extracted.strip() == str(answer).strip():
                    return 1.0
                return -0.5  # Wrong answer penalty

        def physics_reward(prompt, completion, answer):
            """Verify physics answer (numeric + units check)."""
            extracted = extract_boxed_answer(completion)
            if not extracted or not answer:
                return 0.0
            try:
                pred_num = float(re.findall(r"[-+]?\d*\.?\d+", extracted)[0])
                gold_num = float(re.findall(r"[-+]?\d*\.?\d+", str(answer))[0])
                if abs(pred_num - gold_num) / max(abs(gold_num), 1e-10) < 0.05:
                    return 1.0
                return -0.5  # Wrong answer penalty
            except (ValueError, IndexError):
                if extracted.strip() == str(answer).strip():
                    return 1.0
                return -0.5  # Wrong answer penalty

        def chemistry_reward(prompt, completion, answer):
            """Verify chemistry answer (equation balance, formulas)."""
            extracted = extract_boxed_answer(completion)
            if not extracted or not answer:
                return 0.0
            if extracted.strip().lower() == str(answer).strip().lower():
                return 1.0
            return -0.5  # Wrong answer penalty

        def generic_reward(prompt, completion, answer):
            """Generic reward for biology/CS (keyword matching + structure)."""
            extracted = extract_boxed_answer(completion)
            if extracted.strip().lower() == str(answer).strip().lower():
                return 1.0
            # Partial credit for containing key terms
            answer_lower = str(answer).lower()
            completion_lower = completion.lower()
            if answer_lower in completion_lower:
                return 0.5
            return -0.5  # Wrong answer penalty

        def conceptual_reward(prompt, completion, answer):
            """Reward for conceptual questions (reasoning quality heuristics)."""
            score = 0.0
            # Reward step-by-step reasoning
            if any(marker in completion.lower() for marker in ["step 1", "first,", "let's think", "because"]):
                score += 0.3
            # Reward Socratic style (asking questions)
            if "?" in completion:
                score += 0.2
            # Reward completeness
            if len(completion.split()) > 50:
                score += 0.2
            # Reward if answer keyword is present
            if answer and str(answer).lower() in completion.lower():
                score += 0.3
            return min(score, 1.0)

        if reward_type == "conceptual":
            return conceptual_reward

        reward_map = {
            "math": math_reward,
            "physics": physics_reward,
            "chemistry": chemistry_reward,
            "biology": generic_reward,
            "cs": generic_reward,
        }
        return reward_map.get(domain, generic_reward)

# Build domain reward functions
verifiable_reward_fns = {d: make_reward_fn(d, "verifiable") for d in DOMAINS}
conceptual_reward_fns = {d: make_reward_fn(d, "conceptual") for d in DOMAINS}
print("Reward functions ready for all domains")

In [ ]:
# ============================================================
# Load model + SFT adapter, prepare for GSPO
# ============================================================
import torch
import json
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import random

# Load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_COMPLETION + MAX_PROMPT_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

# ---- Load SFT adapter weights into a fresh LoRA ----
# NOTE: We cannot use merge_and_unload() — it breaks Unsloth's pre-allocated
# inference buffers and causes "Expected dtype float, got BFloat16" errors.
# Instead: read SFT adapter config, create fresh LoRA with same architecture,
# then load the SFT weights via load_state_dict(strict=False).

# Read SFT adapter config to match LoRA architecture
sft_config_path = os.path.join(SFT_CHECKPOINT, "adapter_config.json")
if os.path.exists(sft_config_path):
    with open(sft_config_path) as f:
        sft_cfg = json.load(f)
    sft_r = sft_cfg.get("r", LORA_R)
    sft_alpha = sft_cfg.get("lora_alpha", LORA_ALPHA)
    print(f"SFT adapter config: r={sft_r}, alpha={sft_alpha}")
else:
    sft_r = LORA_R
    sft_alpha = LORA_ALPHA
    print(f"No SFT adapter_config.json found, using defaults: r={sft_r}, alpha={sft_alpha}")

# Use max(sft_r, LORA_R) to accommodate SFT weights
effective_r = max(sft_r, LORA_R)

# Create fresh LoRA via Unsloth (preserves gradient hooks)
model = FastLanguageModel.get_peft_model(
    model,
    r=effective_r,
    lora_alpha=sft_alpha,
    lora_dropout=0.05,
    target_modules=TARGET_MODULES,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Load SFT adapter weights
from safetensors.torch import load_file
sft_weights_path = os.path.join(SFT_CHECKPOINT, "adapter_model.safetensors")
if os.path.exists(sft_weights_path):
    sft_weights = load_file(sft_weights_path)
    incompatible = model.load_state_dict(sft_weights, strict=False)
    print(f"Loaded SFT weights from {sft_weights_path}")
    if incompatible.unexpected_keys:
        print(f"  Unexpected keys (ignored): {len(incompatible.unexpected_keys)}")
    print(f"  Missing keys (fresh init): {len(incompatible.missing_keys)}")
else:
    print(f"WARNING: SFT weights not found at {sft_weights_path}, starting from base model")

# Fix device map to prevent gradient offloading
if hasattr(model, 'hf_device_map'):
    model.hf_device_map = {'': 0}

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

SYSTEM_PROMPT = "Ты — репетитор по STEM. Реши задачу пошагово и запиши финальный ответ в \\boxed{}."

def create_grpo_reward_fn(reward_fns_map, problems_list):
    """Create a unified reward function for GRPOTrainer."""
    prompt_to_problem = {}
    for p in problems_list:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": p["prompt"]},
        ]
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt_to_problem[formatted_prompt.strip()] = p

    def reward_fn(completions, prompts=None, **kwargs):
        """Reward function matching TRL GRPOTrainer signature.

        TRL passes completions as the first positional argument.
        prompts is passed as keyword or second positional arg.
        """
        if prompts is None:
            prompts = [""] * len(completions)
        rewards = []
        for prompt_text, completion_text in zip(prompts, completions):
            problem = prompt_to_problem.get(prompt_text.strip())
            if problem is None:
                rewards.append(0.0)
                continue
            domain = problem.get("domain", "math")
            answer = problem.get("answer", "")
            fn = reward_fns_map.get(domain, reward_fns_map.get("math"))
            reward = fn(prompt_text, completion_text, answer)
            rewards.append(reward)
        return rewards

    return reward_fn


def format_problems_as_dataset(problem_list):
    """Format problems into a Dataset for GRPOTrainer."""
    formatted = []
    for p in problem_list:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": p["prompt"]},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        formatted.append({"prompt": prompt})
    return Dataset.from_list(formatted)


print("Model and reward functions ready for GSPO training")

In [ ]:
# ============================================================
# Training: Phase A (verifiable) and Phase B (+ conceptual)
# ============================================================

training_logs = []


def run_gspo_phase(phase_name, problems_list, reward_fns_map, max_steps, output_subdir):
    """Run a GSPO training phase."""
    print(f"\n{'='*60}")
    print(f"GSPO {phase_name}")
    print(f"{'='*60}")
    print(f"  Problems: {len(problems_list)}, Max steps: {max_steps}")

    ds = format_problems_as_dataset(problems_list)
    reward_fn = create_grpo_reward_fn(reward_fns_map, problems_list)

    phase_output = os.path.join(OUTPUT_DIR, output_subdir)
    os.makedirs(phase_output, exist_ok=True)

    gspo_config = GRPOConfig(
        output_dir=phase_output,
        max_steps=max_steps,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        num_generations=G,
        max_completion_length=MAX_COMPLETION,
        max_prompt_length=MAX_PROMPT_LENGTH,
        # GSPO parameters (from Qwen3 paper + TRL docs)
        importance_sampling_level=IMPORTANCE_SAMPLING_LEVEL,
        loss_type=LOSS_TYPE,
        beta=BETA,
        epsilon=EPSILON,
        epsilon_high=EPSILON_HIGH,
        steps_per_generation=STEPS_PER_GENERATION,
        # Standard
        bf16=True,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        optim="adamw_torch_fused",
        max_grad_norm=0.1,
        temperature=0.9,
        seed=42,
        report_to="none",
    )

    trainer = GRPOTrainer(
        model=model,
        args=gspo_config,
        train_dataset=ds,
        reward_funcs=reward_fn,
        processing_class=tokenizer,
    )

    print("Starting GSPO training...")
    result = trainer.train()

    print(f"\n{phase_name} complete! Loss: {result.training_loss:.4f}")
    trainer.save_model(os.path.join(phase_output, "final"))

    training_logs.append({
        "phase": phase_name,
        "steps": max_steps,
        "final_loss": result.training_loss,
        "metrics": result.metrics,
    })
    return trainer, result


# ---- Phase A: Verifiable rewards only ----
trainer_a, result_a = run_gspo_phase(
    "Phase A (Verifiable Only)",
    verifiable_problems,
    verifiable_reward_fns,
    TOTAL_STEPS_PHASE_A,
    "phase_a",
)

# ---- Phase B: Add conceptual rewards ----
combined_reward_fns = {}
for d in DOMAINS:
    v_fn = verifiable_reward_fns[d]
    c_fn = conceptual_reward_fns[d]
    def make_combined(v=v_fn, c=c_fn):
        def combined(prompt, completion, answer):
            return 0.7 * v(prompt, completion, answer) + 0.3 * c(prompt, completion, answer)
        return combined
    combined_reward_fns[d] = make_combined()

all_problems = verifiable_problems + conceptual_problems
random.seed(42)
random.shuffle(all_problems)

trainer_b, result_b = run_gspo_phase(
    "Phase B (Verifiable + Conceptual)",
    all_problems,
    combined_reward_fns,
    TOTAL_STEPS_PHASE_B,
    "phase_b",
)

print("\n" + "="*60)
print("GSPO Training Summary")
for log in training_logs:
    print(f"  {log['phase']}: loss={log['final_loss']:.4f}")

In [ ]:
# ============================================================
# Per-domain evaluation
# ============================================================

print("Running per-domain evaluation...")

FastLanguageModel.for_inference(model)

eval_results = defaultdict(lambda: {"rewards": [], "total": 0, "correct": 0})

eval_sample_size = min(50, len(problems))
eval_problems = random.sample(problems, eval_sample_size)

for problem in eval_problems:
    domain = problem.get("domain", "unknown")
    answer = problem.get("answer", "")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem["prompt"]},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_COMPLETION,
            temperature=0.7,
            do_sample=True,
        )

    completion = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    reward_fn = verifiable_reward_fns.get(domain, verifiable_reward_fns.get("math"))
    reward = reward_fn(problem["prompt"], completion, answer)

    eval_results[domain]["rewards"].append(reward)
    eval_results[domain]["total"] += 1
    if reward >= 0.5:
        eval_results[domain]["correct"] += 1

print("\nPer-domain evaluation results:")
overall_rewards = []
for domain in DOMAINS:
    m = eval_results[domain]
    if m["total"] > 0:
        avg_reward = sum(m["rewards"]) / len(m["rewards"])
        accuracy = 100 * m["correct"] / m["total"]
        print(f"  {domain}: avg_reward={avg_reward:.3f}, accuracy={accuracy:.1f}% ({m['correct']}/{m['total']})")
        overall_rewards.extend(m["rewards"])

if overall_rewards:
    print(f"\n  Overall: avg_reward={sum(overall_rewards)/len(overall_rewards):.3f}")

eval_path = os.path.join(OUTPUT_DIR, "gspo_eval_metrics.json")
with open(eval_path, "w") as f:
    json.dump({
        "domain_results": {d: {"avg_reward": sum(v["rewards"])/max(len(v["rewards"]),1), "accuracy": v["correct"]/max(v["total"],1), "total": v["total"]} for d, v in eval_results.items()},
        "training_logs": training_logs,
        "config": {"G": G, "beta": BETA, "LR": LEARNING_RATE, "loss_type": LOSS_TYPE},
    }, f, indent=2, default=str)
print(f"Evaluation saved to {eval_path}")

In [ ]:
# ============================================================
# Save final adapter
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final GSPO adapter saved to {final_adapter_path}")

config_to_save = {
    "stage": "gspo",
    "pipeline": "SFT -> GSPO (SimPO skipped)",
    "base_model": BASE_MODEL,
    "sft_checkpoint": SFT_CHECKPOINT,
    "importance_sampling_level": IMPORTANCE_SAMPLING_LEVEL,
    "loss_type": LOSS_TYPE,
    "beta": BETA,
    "epsilon": EPSILON,
    "epsilon_high": EPSILON_HIGH,
    "steps_per_generation": STEPS_PER_GENERATION,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "G": G,
    "max_completion": MAX_COMPLETION,
    "learning_rate": LEARNING_RATE,
    "phase_a_steps": TOTAL_STEPS_PHASE_A,
    "phase_b_steps": TOTAL_STEPS_PHASE_B,
    "total_problems": len(problems),
    "verifiable_problems": len(verifiable_problems),
    "conceptual_problems": len(conceptual_problems),
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config_to_save, f, indent=2)
print(f"Config saved to {config_path}")

# Optional: push to HF
PUSH_TO_HUB = False
HF_REPO_ID = "Siesher/mits-qwen3-4b-gspo"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

print("\nDone! GSPO adapter ready for STaR (next stage).")